In [1]:
import pandas as pd

In [8]:
# Load files
pheno = pd.read_csv("../data/final_dataset/height_adj.txt", sep="\s+", header=None, names=["FID", "IID", "Height"])
# Load metadata safely, keeping only user + inferred_sex columns
meta = pd.read_csv(
    "../data/final_dataset/gwas_firstrun/metadata.txt",
    sep="\t",  # or try delim_whitespace=True if tabs don’t work
    usecols=["user", "inferred_sex"],
    dtype=str  # prevent type conversion errors
)

# Rename to match for merging
meta = meta.rename(columns={"user": "IID", "inferred_sex": "Sex"})
# Convert ID to same type (int or str)
pheno["IID"] = pheno["IID"].astype(str)
meta["IID"] = meta["IID"].astype(str)

# Merge by IID
covariates = pheno[["FID", "IID"]].merge(meta, on="IID")

# Replace missing values in Sex with "NA" or "Undefined"
covariates["Sex"] = covariates["Sex"].fillna("NA")  # or .fillna("Undefined")

# Keep only unique rows by FID+IID
covariates = covariates.drop_duplicates(subset=['FID', 'IID'])

# Save covariate file
covariates.to_csv("../data/final_dataset/gwas_firstrun/covariates.txt", sep="\t", index=False)

In [10]:
print(pheno.head())

     FID    IID  Height
0  11819  11819   173.0
1  11802  11802   168.0
2  11744  11744   185.0
3  11739  11739   181.0
4  11698  11698   175.0


In [13]:
# Check how many missing (NA) values are in column 3 (index 2)
pheno["Height"].isna().sum()

0

In [14]:
## error search

In [18]:
# Load fam file (plink .fam format: FID, IID, ...)
fam = pd.read_csv("../data/final_dataset/afterqc.fam", delim_whitespace=True, header=None, usecols=[0,1], names=["FID", "IID"])

# Load phenotype file (adjust names accordingly)
pheno = pd.read_csv("../data/final_dataset/height_adj.txt", delim_whitespace=True, header=None, names=["FID", "IID", "Height"])

# Load covariates file
covar = pd.read_csv("../data/final_dataset/gwas_firstrun/covariates.txt", sep="\t")

# Check sample ID overlap
print(f"Samples in fam but not in pheno: {(set(fam['IID']) - set(pheno['IID']))}")
print(f"Samples in pheno but not in fam: {(set(pheno['IID']) - set(fam['IID']))}")

print(f"Samples in fam but not in covar: {(set(fam['IID']) - set(covar['IID']))}")
print(f"Samples in covar but not in fam: {(set(covar['IID']) - set(fam['IID']))}")

Samples in fam but not in pheno: set()
Samples in pheno but not in fam: {8203, 8204, 14, 8206, 17, 10260, 2070, 8217, 26, 35, 6182, 10279, 6184, 6189, 8238, 60, 10302, 64, 4170, 6221, 2133, 10325, 86, 4198, 10347, 10348, 6263, 8314, 8319, 10383, 2199, 6296, 158, 8360, 4271, 8369, 6321, 180, 10424, 187, 10430, 8383, 4292, 8393, 203, 10448, 2274, 6373, 8423, 2287, 10490, 8449, 262, 4360, 8463, 8466, 2322, 10523, 288, 10530, 8483, 305, 10559, 4428, 10575, 8529, 339, 6483, 8534, 10586, 8538, 6493, 2398, 352, 8546, 8554, 8561, 8563, 10620, 10632, 10635, 6545, 8598, 10653, 6560, 6583, 8635, 4546, 2506, 10702, 8655, 8656, 8666, 4577, 8675, 4583, 6633, 4585, 6639, 4604, 512, 4613, 6665, 8715, 6667, 10770, 10773, 8731, 8734, 10785, 8739, 4644, 10790, 6695, 6699, 4667, 579, 580, 6725, 8780, 8783, 8787, 6745, 8803, 2660, 2662, 8808, 8818, 10868, 637, 10883, 2691, 10885, 644, 8840, 6793, 651, 8844, 2705, 4754, 8851, 6805, 6815, 8864, 10919, 6828, 2732, 693, 10936, 2746, 703, 704, 10948, 8905, 717,

In [20]:
# Extract sample IDs from fam file to filter on
fam_ids = set(fam["FID"])

# Filter phenotype to keep only samples in fam_ids
pheno_filtered = pheno[pheno["FID"].isin(fam_ids)].copy()

# Filter covariates to keep only samples in fam_ids
covar_filtered = covar[covar["FID"].isin(fam_ids)].copy()

# Print info to check filtering
print(f"Original pheno samples: {len(pheno)}")
print(f"Filtered pheno samples: {len(pheno_filtered)}\n")

print(f"Original covariate samples: {len(covar)}")
print(f"Filtered covariate samples: {len(covar_filtered)}")

Original pheno samples: 1106
Filtered pheno samples: 647

Original covariate samples: 1071
Filtered covariate samples: 645


In [21]:
# check sample overlap between phenotypes and covariates
common_ids = set(pheno_filtered["FID"]).intersection(set(covar_filtered["FID"]))
print(f"Number of samples in both pheno and covar after filtering: {len(common_ids)}")

Number of samples in both pheno and covar after filtering: 645


In [22]:
# keep only samples present in both, phenotype and covariates
pheno_final = pheno_filtered[pheno_filtered["FID"].isin(common_ids)].copy()
covar_final = covar_filtered[covar_filtered["FID"].isin(common_ids)].copy()

print(f"Final pheno samples: {len(pheno_final)}")
print(f"Final covariate samples: {len(covar_final)}")

Final pheno samples: 647
Final covariate samples: 645
